### 2.8 DeepSpeed训练CLI与训练成果展示

开始训练 ↓

<center><img src="http://ml2022.oss-cn-hangzhou.aliyuncs.com/img/0d46acea891701fd573115782288e05.jpg" alt="0d46acea891701fd573115782288e05" style="zoom:50%;" />

经过18小时的训练，得到如下结果 ↓

- wandb显示状态——

<center><img src="http://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20241023195052407.png" alt="image-20241023195052407" style="zoom:33%;" />

- CLI显示状态——

<center><img src="http://ml2022.oss-cn-hangzhou.aliyuncs.com/img/9f1b2321dc79550b1e0f194e165ff22.png" alt="9f1b2321dc79550b1e0f194e165ff22" style="zoom:50%;" />

我们会在目录下看到 ↓

![](http://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/102.png)

- pretrain_512.pth：模型权重参数（网络的权重和偏置）、用于保存和加载模型权重，适用于推理、继续训练

- pretrain_512_optimizer.pth：用于保存优化器状态（学习率、动量、梯度累积状态等）、用于恢复优化器状态，适用于继续训练

**为什么要同时保存模型和优化器状态**？

在训练过程中，优化器会根据模型的梯度来调整权重，如果你在训练中断后只恢复了模型的权重，而没有恢复优化器的状态，优化器的学习率、动量等信息将会丢失，可能会导致继续训练时表现不佳，或者无法顺利收敛。因此，为了保证训练的连续性，继续训练时需要同时加载模型权重和优化器状态。而在推理或测试模型时，只需要加载模型权重文件就足够了。

接下来我们可以用下面的代码来测试模型运行的效果 ↓ 这段代码被我们打包在了`inference.ipy`中，你可以在线上jupyter中运行。

- 训练结束后测试运行

In [ ]:
import itertools
import re
import json
import jsonlines
import psutil
import ujson
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from datasets import load_dataset
import os
from tqdm import tqdm
import torch
from model.model import Transformer  # 确保路径正确
from model.LMConfig import LMConfig   # 导入 LMConfig

In [ ]:
# 定义BOS和EOS标记
bos_token = "<s>"
eos_token = "</s>"

In [ ]:
# 加载训练好的分词器路径
tokenizer = AutoTokenizer.from_pretrained('/root/autodl-tmp/MateConv/model/mateconv_tokenizer', use_fast=False)
print(f'加载的tokenizer词表大小: {len(tokenizer)}')

In [211]:
# 创建配置对象
lm_config = LMConfig()

In [212]:
# 初始化 Transformer 模型
model = Transformer(lm_config)

In [213]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [214]:
device

device(type='cuda')

In [215]:
model.to(device)

# 检查模型结构和参数
print(model)

Transformer(
  (tok_embeddings): Embedding(6400, 512)
  (dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlock(
      (attention): Attention(
        (wq): Linear(in_features=512, out_features=512, bias=False)
        (wk): Linear(in_features=512, out_features=256, bias=False)
        (wv): Linear(in_features=512, out_features=256, bias=False)
        (wo): Linear(in_features=512, out_features=512, bias=False)
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
      (feed_forward): FeedForward(
        (w1): Linear(in_features=512, out_features=1408, bias=False)
        (w2): Linear(in_features=1408, out_features=512, bias=False)
        (w3): Linear(in_features=512, out_features=1408, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=512, 

In [217]:
# 加载模型权重【这里要修改为你的模型地址】
model.load_state_dict(torch.load('out/pretrain_512.pth', map_location=device))
model.eval()  # 切换到评估模式

Transformer(
  (tok_embeddings): Embedding(6400, 512)
  (dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlock(
      (attention): Attention(
        (wq): Linear(in_features=512, out_features=512, bias=False)
        (wk): Linear(in_features=512, out_features=256, bias=False)
        (wv): Linear(in_features=512, out_features=256, bias=False)
        (wo): Linear(in_features=512, out_features=512, bias=False)
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
      (feed_forward): FeedForward(
        (w1): Linear(in_features=512, out_features=1408, bias=False)
        (w2): Linear(in_features=1408, out_features=512, bias=False)
        (w3): Linear(in_features=512, out_features=1408, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=512, 

In [43]:
# 准备输入文本
input_text = "决策树是机器学习中的一种算法，决策树是"
input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

# 生成文本
max_new_tokens = 500  # 生成 100 个 token
eos_token_id = tokenizer.eos_token_id  # 终止 token

# 在model.py中我们定义了generate方法、专用于生成
# 调用 model.generate() 进行生成
output_ids = next(model.generate(
    idx=input_ids,
    eos=eos_token_id,
    max_new_tokens=max_new_tokens,
    temperature=0.2,  # 控制创造性
    top_k=10,  # 限制 top-k 采样
    rp=1.2,  # 避免重复
    stream=False  # 关闭流式返回
))

# 解码生成的 token
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 打印最终生成的文本
print(generated_text)

基于人类数据的计算方法。
在本书中，作者通过对比分析了大脑和大脑之间的关系、思维方式以及行为模式等问题，并结合大量的数据资料进行了深入剖析，为读者提供了一个很好的参考工具。


In [45]:
# 准备输入文本
input_text = "我与你"
input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

# 生成文本
max_new_tokens = 500  # 生成 100 个 token
eos_token_id = tokenizer.eos_token_id  # 终止 token

# 在model.py中我们定义了generate方法、专用于生成
# 调用 model.generate() 进行生成
output_ids = next(model.generate(
    idx=input_ids,
    eos=eos_token_id,
    max_new_tokens=max_new_tokens,
    temperature=1,  # 控制创造性
    top_k=10,  # 限制 top-k 采样
    rp=1.2,  # 避免重复
    stream=False  # 关闭流式返回
))

# 解码生成的 token
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 打印最终生成的文本
print(generated_text)

在一起。
我爱你，也许你会觉得有点不公平了，但我会努力奋斗一辈子！我爱你，也许只是一段美好的时光而已。”他如此的坚定信念：“对不起，我的爱，是我的力量。希望能帮助你的人生，让生命更加美丽！”
“我们是最好的男人！”，他说到最后。“如果有一天你不会再见过我，我一定会珍惜这份温暖和关怀。


### 2.9【选学】借助wandb进行训练过程记录

在大规模模型训练中，我们往往需要监控和分析大量的训练数据，而WandB可以帮助我们实现这一目标。它提供了以下几个重要的功能——

**实时可视化**：WandB可以实时展示训练过程中关键指标的变化，如损失函数、学习率、训练时间等。通过这些可视化数据，我们能够直观地了解模型的训练进展，快速发现训练中的异常或瓶颈。

**自动记录与日志管理**：WandB会自动记录每次实验的参数、代码、输出结果，确保实验结果的可追溯性。无论是超参数的设置，还是模型的架构调整，WandB都能够帮助我们完整保留实验记录，方便后期对比与调优。

**支持中断与恢复训练**：在长时间的预训练任务中，系统中断或需要暂停是常见的情况。通过WandB的checkpoint功能，我们可以随时恢复训练，从上次中断的地方继续进行，避免数据和时间的浪费。

**多实验对比**：当我们尝试不同的模型配置或超参数时，WandB允许我们在多个实验之间轻松进行对比分析，帮助我们选择最优的模型配置。

**团队协作**：WandB还支持团队协作，多个成员可以共同查看实验结果，协同调试模型。这对研究和项目开发中团队的合作非常有帮助。

---

要使用WandB，你需遵循下面的流程 ↓

1. 注册你的WandB账号：http://wandb.ai/site

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/94.png)
![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/95.png)
![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/96.png)
![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/97.png)

获取你的API ↓

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/98.png)

2. 在CLI安装并使用wandb

在命令行中输入如下代码安装wandb ↓

```shell
#安装
pip install wandb

#进入你pretrain所在的虚拟环境
conda activate MateConv
cd ~/autodl-tmp/MateConv

#登录wandb
wandb login
```

你需要根据提示输入你的API-KEY：

![](https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/99.png)

即可在当前电脑上保存wandb账号信息，之后即可直接在wandb home主页上看到训练过程。

3. 借助wandb监控当前运行效果

接下来在命令行中尝试运行该指令，该指令是在执行deepspeed命令的同时让wandb完成监控，这个命令中没有显示设置epoch数量，当然你也可以将epoch数量补充进来。注意由于我们本质执行的是deepspeed，所以pretrain.py执行前需要先进入pretrain.py所在的具体目录 ↓

```shell
torch pretrain.py

deepspeed --num_gpus=4 pretrain.py --use_wandb --wandb_project "MateConv-Pretrain" --deepspeed ds_config.json
```

你会看到这里返回的链接 ↓

![](http://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/100.png)

点击链接后即可进入模型监控页面 ↓

![](http://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/training/101.png)